# NovaByte Retail S.L. - Automatización y Auditoría de Inventarios
### Pipeline Dinámico con Detección de Outliers por IQR y Separación de Datasets
**Módulo M3 · Programació d'Intel·ligència Artificial (RA1)**

---

## 📌 Enfoque de Automatización Reutilizable (Agnóstico a la Empresa)
En un sistema de automatización profesional **no se deben prefijar valores fijos (*hardcoded*)** como `stock == 99999` o rangos específicos de un único almacén. Mañana podría llegar otra empresa con productos de otra naturaleza (tornillos con miles de unidades o maquinaria pesada con 3 unidades).

Por este motivo, se implementa una **solución estadística adaptativa basada en IQR (Rango Intercuartílico)**:
1. **Carga completa sin pérdida de datos**: Se ingieren todos los registros en bruto (**CSV** y **JSON**) manteniendo la trazabilidad del centro logístico.
2. **Cálculo dinámico de límites IQR**:
   * $IQR = Q_3 - Q_1$
   * Límite superior moderado: $Q_3 + 1.5 \times IQR$
   * Límite superior extremo: $Q_3 + 3.0 \times IQR$
3. **Segmentación en Dos Datasets Independientes**:
   * **Dataset Procesable (`df_procesable`)**: Registros limpios y dentro del rango estadístico normal para calcular balances, medias y reposiciones reales.
   * **Dataset de Cuarentena (`df_cuarentena_outliers`)**: Registros sospechosamente altos (outliers) y con anomalías de validación, aislados con una columna explicativa (`motivo_cuarentena`) para revisión humana.
4. **Exportación automática** de ambos datasets en **CSV** y **JSON**.

> **Preparado para Google Colab y Jupyter**: Funciona en ambos entornos sin dependencias complejas.

In [ ]:
# =============================================================================
# 0. Configuración del entorno y librerías
# =============================================================================
import os
import json
import pandas as pd
import numpy as np

# Compatibilidad para visualización display()
try:
    from IPython.display import display
except ImportError:
    display = print

# Gráficos
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    sns.set_theme(style="whitegrid", palette="muted")
    plt.rcParams["figure.figsize"] = (12, 6)
    plt.rcParams["font.size"] = 11
    HAS_PLT = True
except ImportError:
    HAS_PLT = False
    print("ℹ️ matplotlib/seaborn no detectados. Puedes instalarlos con: pip install matplotlib seaborn")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

# En Google Colab, descomenta para subir los archivos manualmente si es necesario:
# from google.colab import files
# print("Sube los archivos: inventario_BCN.csv, inventario_MAD.csv, inventario_VAL.json, inventario_SEV.json")
# uploaded = files.upload()

print("✅ Entorno preparado correctamente.")

---
## 📥 1. Carga Dinámica de Datos en Bruto (Sin Borrado de Registros)

Cargamos los archivos CSV y JSON preservando el 100% de los datos originales.

In [ ]:
def cargar_csv_agnostico(ruta: str, centro: str) -> pd.DataFrame:
    """Carga un archivo CSV sin descarte previo de datos."""
    if not os.path.exists(ruta):
        raise FileNotFoundError(f"Archivo no encontrado: {ruta}")
    df = pd.read_csv(ruta, encoding="utf-8")
    df["centro_origen"] = centro
    df["formato_origen"] = "CSV"
    df["archivo_origen"] = os.path.basename(ruta)
    return df

def cargar_json_agnostico(ruta: str, centro: str) -> pd.DataFrame:
    """Carga un archivo JSON sin descarte previo de datos."""
    if not os.path.exists(ruta):
        raise FileNotFoundError(f"Archivo no encontrado: {ruta}")
    with open(ruta, "r", encoding="utf-8") as f:
        data = json.load(f)
    df = pd.DataFrame(data)
    df["centro_origen"] = centro
    df["formato_origen"] = "JSON"
    df["archivo_origen"] = os.path.basename(ruta)
    return df

# Carga de los 4 almacenes de NovaByte Retail
df_bcn = cargar_csv_agnostico("inventario_BCN.csv", "Barcelona")
df_mad = cargar_csv_agnostico("inventario_MAD.csv", "Madrid")
df_val = cargar_json_agnostico("inventario_VAL.json", "Valencia")
df_sev = cargar_json_agnostico("inventario_SEV.json", "Sevilla")

# Unificación en un DataFrame Maestro (3.000 registros completos)
df_raw = pd.concat([df_bcn, df_mad, df_val, df_sev], ignore_index=True)

print(f"📦 Total registros brutos cargados: {len(df_raw)} filas, {df_raw.shape[1]} columnas.")
display(df_raw.head())

---
## 📊 2. Análisis Descriptivo Global de Todos los Productos

Evaluamos la estructura completa del inventario para conocer volúmenes, categorías y distribuciones.

In [ ]:
# Normalización segura de tipos a numéricos para auditoría
df_raw["precio_num"] = pd.to_numeric(df_raw["precio_unitario"], errors="coerce")
df_raw["stock_num"] = pd.to_numeric(df_raw["stock_actual"], errors="coerce")
df_raw["stock_min_num"] = pd.to_numeric(df_raw["stock_minimo"], errors="coerce")

print("=== ESTADÍSTICAS GLOBALES EN BRUTO ===")
display(df_raw[["precio_num", "stock_num", "stock_min_num"]].describe())

if HAS_PLT:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    sns.countplot(data=df_raw, x="centro_origen", hue="formato_origen", ax=axes[0], palette="Set2")
    axes[0].set_title("Volumen de Registros por Centro Logístico", fontsize=13, fontweight="bold")
    
    cat_counts = df_raw["categoria"].astype(str).str.capitalize().value_counts()
    sns.barplot(x=cat_counts.values, y=cat_counts.index, ax=axes[1], palette="mako")
    axes[1].set_title("Frecuencia de Productos por Categoría", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("\nConteo de categorías:")
    print(df_raw["categoria"].astype(str).str.capitalize().value_counts())

---
## 📐 3. Detección Dinámica de Outliers Mediante IQR (Rango Intercuartílico)

El método del **Rango Intercuartílico (IQR)** calcula las barreras estadísticas en base a la dispersión real de los datos:

* $Q_1$ (percentil 25): Primer cuartil.
* $Q_3$ (percentil 75): Tercer cuartil.
* $IQR = Q_3 - Q_1$: Dispersión intercuartílica.
* **Límite Superior Moderado**: $Q_3 + 1.5 \times IQR$
* **Límite Superior Extremo**: $Q_3 + 3.0 \times IQR$

Cualquier producto que supere estos límites es catalogado como **outlier estadístico sospechoso** y se traslada al dataset de auditoría de forma dinámica, sin depender de constantes fijas.

In [ ]:
def calcular_umbrales_iqr(serie: pd.Series, factor_moderado: float = 1.5, factor_extremo: float = 3.0) -> dict:
    """
    Calcula dinámicamente los umbrales IQR sobre los valores numéricos válidos (>= 0).
    Es totalmente agnóstico al dominio de datos de cualquier empresa.
    """
    s_val = serie[(serie.notna()) & (serie >= 0)]
    if len(s_val) == 0:
        return {}
    
    q1 = float(s_val.quantile(0.25))
    q3 = float(s_val.quantile(0.75))
    iqr = q3 - q1
    
    return {
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "limite_inf": max(0.0, q1 - factor_moderado * iqr),
        "limite_sup_moderado": q3 + factor_moderado * iqr,
        "limite_sup_extremo": q3 + factor_extremo * iqr
    }

# Cálculo dinámico sobre las variables numéricas del dataset
stats_stock = calcular_umbrales_iqr(df_raw["stock_num"])
stats_precio = calcular_umbrales_iqr(df_raw["precio_num"])

print("=== UMBRALES DINÁMICOS DETECTADOS POR IQR ===")
print(f"📦 STOCK ACTUAL:")
print(f"   - Q1 (25%): {stats_stock['q1']:.2f} | Q3 (75%): {stats_stock['q3']:.2f} | IQR: {stats_stock['iqr']:.2f}")
print(f"   - Límite sospechoso moderado (1.5 x IQR): > {stats_stock['limite_sup_moderado']:.2f} unidades")
print(f"   - Límite sospechoso extremo  (3.0 x IQR): > {stats_stock['limite_sup_extremo']:.2f} unidades")

print(f"\n💶 PRECIO UNITARIO:")
print(f"   - Q1 (25%): {stats_precio['q1']:.2f} € | Q3 (75%): {stats_precio['q3']:.2f} € | IQR: {stats_precio['iqr']:.2f} €")
print(f"   - Límite sospechoso moderado (1.5 x IQR): > {stats_precio['limite_sup_moderado']:.2f} €")
print(f"   - Límite sospechoso extremo  (3.0 x IQR): > {stats_precio['limite_sup_extremo']:.2f} €")

---
## 🛡️ 4. Función de Segmentación Automatizada: División en Dos Datasets

Construimos la función modular `clasificar_y_separar_inventario` que analiza cada registro y lo dirige a:
1. **`df_procesable`**: Datos válidos, dentro de rangos operativos normales.
2. **`df_cuarentena_outliers`**: Datos con anomalías (outliers IQR extremos, negativos, nulos o duplicados), adjuntando una columna con el **motivo de sospecha**.

In [ ]:
def clasificar_y_separar_inventario(df: pd.DataFrame, stats_stock: dict, stats_precio: dict):
    """
    Segmenta un DataFrame en dos conjuntos independientes:
    1. Dataset procesable (operativo).
    2. Dataset de cuarentena (outliers y errores).
    """
    registros_limpios = []
    registros_cuarentena = []
    
    # Detección de duplicados en producto_id
    ids_validos = df["producto_id"].dropna().astype(str).str.strip()
    ids_duplicados = set(ids_validos[ids_validos.duplicated()].unique())
    
    for idx, row in df.iterrows():
        motivos_cuarentena = []
        pid = str(row["producto_id"]).strip() if pd.notna(row["producto_id"]) else ""
        precio = row["precio_num"]
        stock = row["stock_num"]
        stock_min = row["stock_min_num"]
        
        # 1. Validación de Identificador
        if not pid:
            motivos_cuarentena.append("ID ausente o vacío")
        elif pid in ids_duplicados:
            motivos_cuarentena.append(f"ID duplicado en catálogo ({pid})")
            
        # 2. Validación y Outliers de Precio
        if pd.isna(precio):
            motivos_cuarentena.append("Precio nulo o no numérico")
        elif precio < 0:
            motivos_cuarentena.append(f"Precio negativo imposible ({precio} €)")
        elif stats_precio and precio > stats_precio.get("limite_sup_extremo", float("inf")):
            motivos_cuarentena.append(f"Precio outlier extremo IQR (> {stats_precio['limite_sup_extremo']:.2f} €)")
            
        # 3. Validación y Outliers de Stock
        if pd.isna(stock):
            motivos_cuarentena.append("Stock actual nulo o no numérico")
        elif stock < 0:
            motivos_cuarentena.append(f"Stock negativo imposible ({stock} uds)")
        elif stats_stock and stock > stats_stock.get("limite_sup_extremo", float("inf")):
            motivos_cuarentena.append(f"Stock sospechosamente alto / Outlier IQR (> {stats_stock['limite_sup_extremo']:.2f} uds, valor={stock})")
            
        # 4. Validación de Stock Mínimo
        if pd.isna(stock_min) or stock_min < 0:
            motivos_cuarentena.append(f"Stock mínimo inválido ({stock_min})")
            
        # Enrutamiento al dataset correspondiente
        row_dict = row.to_dict()
        if motivos_cuarentena:
            row_dict["motivo_cuarentena"] = " | ".join(motivos_cuarentena)
            registros_cuarentena.append(row_dict)
        else:
            registros_limpios.append(row_dict)
            
    df_procesable = pd.DataFrame(registros_limpios)
    df_cuarentena = pd.DataFrame(registros_cuarentena)
    return df_procesable, df_cuarentena

# Ejecución de la separación
df_procesable, df_cuarentena = clasificar_y_separar_inventario(df_raw, stats_stock, stats_precio)

print(f"✅ SEPARACIÓN DE DATASETS COMPLETADA:")
print(f" 🔹 Dataset Procesable (Operativo):      {len(df_procesable)} registros ({len(df_procesable)/len(df_raw)*100:.1f}%)")
print(f" 🔸 Dataset de Cuarentena (Sospechosos): {len(df_cuarentena)} registros ({len(df_cuarentena)/len(df_raw)*100:.1f}%)")

---
## 🔬 5. Auditoría del Dataset de Cuarentena (Registros Aislados)

Revisamos los productos que fueron apartados por ser sospechosamente altos o contener errores.

In [ ]:
print("=== DESGLOSE DE MOTIVOS DE AISLAMIENTO EN CUARENTENA ===")
display(df_cuarentena["motivo_cuarentena"].value_counts())

print("\n=== DETALLE DE PRODUCTOS SOSPECHOSAMENTE ALTOS POR IQR (STOCK 99.999 Y OUTLIERS) ===")
outliers_iqr = df_cuarentena[df_cuarentena["motivo_cuarentena"].str.contains("Outlier IQR")].copy()
cols_vista = ["centro_origen", "producto_id", "nombre", "categoria", "stock_actual", "precio_unitario", "motivo_cuarentena"]
display(outliers_iqr[cols_vista])

In [ ]:
# Visualización del Impacto de la Separación: Antes vs Después
if HAS_PLT:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    # Gráfico 1: Dataset en Bruto (sesgado por outliers)
    sns.boxplot(x="centro_origen", y="stock_num", data=df_raw, ax=axes[0], palette="Reds")
    axes[0].set_title("Inventario EN BRUTO (Con Outliers de 99.999 uds)", fontsize=12, fontweight="bold", color="darkred")
    axes[0].set_ylabel("Stock Actual")
    
    # Gráfico 2: Dataset Procesable (tras aislamiento por IQR)
    sns.boxplot(x="centro_origen", y="stock_num", data=df_procesable, ax=axes[1], palette="Greens")
    axes[1].set_title("Dataset PROCESABLE (Filtrado Dinámico con IQR)", fontsize=12, fontweight="bold", color="darkgreen")
    axes[1].set_ylabel("Stock Actual")
    
    plt.tight_layout()
    plt.show()

# Impacto financiero real
val_bruto = (df_raw["precio_num"] * df_raw["stock_num"]).sum()
val_operativo = (df_procesable["precio_num"] * df_procesable["stock_num"]).sum()

print(f"💰 IMPACTO EN LA VALORACIÓN EMPRESARIAL:")
print(f" - Valoración en bruto (contaminada por 99999): {val_bruto:,.2f} €")
print(f" - Valoración real procesable (operativa):        {val_operativo:,.2f} €")
print(f" - Distorsión evitada gracias al aislamiento:     {(val_bruto - val_operativo):,.2f} €")

---
## 💾 6. Exportación Automática de los Datasets Separados

Guardamos los resultados en archivos independientes para su posterior uso en operaciones y auditoría:
* `inventario_procesable.csv` y `.json`: Para las operaciones del día a día (reposición, valoración, estadísticas).
* `inventario_cuarentena_outliers.csv` y `.json`: Para el equipo de calidad de datos y almacén.

In [ ]:
# Exportación en CSV y JSON
# Usamos el directorio actual o carpeta de salida
output_dir = "."

# 1. Guardar Dataset Procesable
path_proc_csv = os.path.join(output_dir, "inventario_procesable.csv")
path_proc_json = os.path.join(output_dir, "inventario_procesable.json")
df_procesable.to_csv(path_proc_csv, index=False, encoding="utf-8")
df_procesable.to_json(path_proc_json, orient="records", indent=2, force_ascii=False)

# 2. Guardar Dataset de Cuarentena / Outliers
path_cuar_csv = os.path.join(output_dir, "inventario_cuarentena_outliers.csv")
path_cuar_json = os.path.join(output_dir, "inventario_cuarentena_outliers.json")
df_cuarentena.to_csv(path_cuar_csv, index=False, encoding="utf-8")
df_cuarentena.to_json(path_cuar_json, orient="records", indent=2, force_ascii=False)

print("📁 ARCHIVOS GENERADOS SATISFACTORIAMENTE:")
print(f" 1. {path_proc_csv} ({len(df_procesable)} filas)")
print(f" 2. {path_proc_json} ({len(df_procesable)} filas)")
print(f" 3. {path_cuar_csv} ({len(df_cuarentena)} filas)")
print(f" 4. {path_cuar_json} ({len(df_cuarentena)} filas)")

---
## 🎯 7. Conclusiones y Ventajas de esta Arquitectura

1. **Reutilización y Generalización Total**:
   * El programa no asume umbrales fijos como `99999` ni un catálogo concreto. Si mañana entra otra empresa con productos farmacéuticos, alimentación o ferretería, el método **IQR calcula los umbrales dinámicamente** en función de la propia distribución de datos recibida.

2. **Seguridad y Trazabilidad (Cero Pérdida de Información)**:
   * No se destruye ningún registro. Los productos anómalos no desaparecen: pasan a un dataset de cuarentena con su causa explícita (`motivo_cuarentena`), permitiendo al equipo de Operaciones pedir explicaciones al almacén correspondiente.

3. **Protección de Métricas de Negocio**:
   * Al aislar los outliers por IQR, la valoración del inventario pasa de más de **400 millones de euros ficticios** a su valor operativo real (~**100.000 €**), permitiendo generar órdenes de reposición y balances comerciales exactos.